# Tarefa 5 - Floresta final e teste único

**Equipe 18** | **Scrum Master da tarefa:** ____________

> O baseline, o rótulo e o corte temporal são os da Tarefa 2. A floresta é a da Tarefa 4 (mesmos hiperparâmetros, mesmas 8 colunas, alvo = classe 12 minutos à frente). Este notebook **não refaz** a ficha de baseline, **não escolhe hiperparâmetros** e **não olha o teste** até você mandar.

## Como usar (3 passos)

1. Rode o notebook inteiro com `ABRIR_TESTE = False` (célula 1). Isso separa os fluxos novos, treina e **congela** a floresta, e mostra as regras. **O arquivo de teste não é lido.**
2. Preencha o diário com parâmetros, colunas, fluxos novos, regras e o critério de ganho sobre a persistência. Depois disso, nada disso muda.
3. Mude `ABRIR_TESTE = True` e rode tudo **uma única vez**. O notebook mede o teste, checa os fluxos novos, salva os resultados e cria um arquivo-trava (`teste_medido.json`) que impede uma segunda medição.

| Entra | Sai |
|---|---|
| `treino.csv`, `validacao.csv`, `baseline_por_fluxo.csv` e o arquivo de teste (Tarefa 2) | Matriz 3x3, F1 por classe, F1 macro, recall de FALHA, comparação com a persistência |
| Hiperparâmetros e colunas da Tarefa 4 | Checagem de fluxo não visto, ficha da árvore e colunas do dicionário v0.4 |

## 0. Bibliotecas, caminhos e configuração

Tudo que define o experimento está nesta célula. **Decida antes de abrir o teste** e não mude depois.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support)
from sklearn.tree import DecisionTreeClassifier, export_text

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)
_v = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
assert _v >= (1, 4), "Atualize o scikit-learn (>= 1.4): a floresta precisa aceitar NaN nas features."

# ============================ CONFIGURAÇÃO ============================
ABRIR_TESTE = False          # só vire True depois de congelar tudo e preencher o diário
REFAZER_CONGELAMENTO = False # só para trocar a configuração ANTES de medir o teste
FORCAR_REMEDIR = False       # depuração; o teste deve ser medido uma única vez

NOME_TESTE = "teste.csv"     # nome do arquivo de teste em data_ripe_atlas/processed

# Hiperparâmetros escolhidos na validação da Tarefa 4 (não buscar de novo)
HIPERPARAMETROS = dict(n_estimators=200, max_depth=10, min_samples_leaf=10,
                       class_weight=None, criterion="gini")
SEMENTE = 42

FEATURES = ["z_robusto_t_menos_1", "delta_z_robusto", "z_robusto", "perda_pct",
            "jitter_relativo", "n5_risco", "n5_timeout", "n5_aumento80"]
ALVO = "status_futuro"       # classe da 3ª medição seguinte (~12 min)
STATUS_ATUAL = "classe"      # só para a comparação de persistência; NUNCA é feature
CLASSES = ["OK", "RISCO", "FALHA"]
NAO_PODEM_ENTRAR = ["pais_origem", "pais_destino", "rota", "dst_addr", "prb_id", "fluxo_id",
                    "avg", "rtt_min_ms", "rtt_max_ms", "jitter_ms", "mediana", "MAD", "jitter_tipico"]

HORIZONTE_SEGUNDOS = 12 * 60
PASSOS_FUTUROS = 3
TOLERANCIA_HORIZONTE_SEGUNDOS = 60

# Fluxos novos (fora do treino). Deixe a lista vazia para a escolha automática.
FLUXOS_NOVOS_MANUAL = []     # ex.: ["6659|103.167.46.8", "6396|212.119.5.254"]
N_LONGO_ESTAVEL = 1
N_CURTO_DEGRADADO = 1
LIMIAR_LONGO_MS = 150
LIMIAR_CURTO_MS = 5

# Critério de ganho sobre a persistência, declarado ANTES do teste
CRITERIO_GANHO = "f1_macro"  # "f1_macro" ou "acuracia_balanceada"
# =====================================================================

CURRENT_DIR = Path.cwd().resolve()
ENTRADAS = ("treino.csv", "validacao.csv", "baseline_por_fluxo.csv")
ROOT = next((c for c in (CURRENT_DIR, *CURRENT_DIR.parents)
             if all((c / "data_ripe_atlas" / "processed" / n).is_file() for n in ENTRADAS)), None)
if ROOT is None:
    raise FileNotFoundError("Não encontrei treino.csv, validacao.csv e baseline_por_fluxo.csv "
                            "em data_ripe_atlas/processed a partir da pasta atual.")
PASTA_SPLITS = ROOT / "data_ripe_atlas" / "processed"
PASTA_MODELOS = ROOT / "data_ripe_atlas" / "models"
PASTA_MODELOS.mkdir(parents=True, exist_ok=True)
CAMINHO_TESTE = PASTA_SPLITS / NOME_TESTE   # NÃO é lido enquanto ABRIR_TESTE = False
ARQ_PARAMS = PASTA_MODELOS / "floresta_final_params.json"
ARQ_TRAVA = PASTA_MODELOS / "teste_medido.json"


def milhar(n):
    return f"{int(n):,}".replace(",", ".")


def br(x, casas=4):
    return f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")


print("Teste aberto nesta execução?", ABRIR_TESTE)

## 1. Dados da Tarefa 2 e verificações de integridade

Lê só treino, validação e a ficha de baseline. Confere que são os blocos da Tarefa 2, que a ficha não mudou e que nenhuma coluna proibida entrou nas features.

- [ ] Treino, validação e teste são os da Tarefa 2
- [ ] O baseline não foi recalculado com o Período B
- [ ] País, IP, rota e RTT absoluto não estão na floresta

In [ ]:
tr = pd.read_csv(PASTA_SPLITS / "treino.csv")
va = pd.read_csv(PASTA_SPLITS / "validacao.csv")
base = pd.read_csv(PASTA_SPLITS / "baseline_por_fluxo.csv")
for d in (tr, va):
    d["timestamp"] = pd.to_numeric(d["timestamp"], errors="raise")

assert set(tr["conjunto"]) == {"treino"} and set(va["conjunto"]) == {"validacao"}
assert set(tr["periodo"]) == {"B"} and set(va["periodo"]) == {"B"}, "Há registro do Período A"
lim_tr = tr.groupby("fluxo_id")["timestamp"].max()
lim_va = va.groupby("fluxo_id")["timestamp"].min()
comuns = lim_tr.index.intersection(lim_va.index)
assert (lim_tr[comuns] < lim_va[comuns]).all(), "Treino e validação se misturam em algum fluxo"
print("Treino N:", milhar(len(tr)), "| Validação N:", milhar(len(va)),
      "| fluxos no treino:", tr["fluxo_id"].nunique())

# A ficha repetida nas linhas precisa ser a do baseline congelado
cols_ficha = ["mediana", "MAD", "jitter_tipico"]
dados = pd.concat([tr, va], ignore_index=True)
assert (dados.groupby("fluxo_id")[cols_ficha].nunique().to_numpy() == 1).all(), "A ficha muda dentro de um fluxo"
cmp = (dados.drop_duplicates("fluxo_id").set_index("fluxo_id")[cols_ficha]
       .join(base.set_index("fluxo_id")[cols_ficha], rsuffix="_baseline", how="left"))
assert cmp.notna().all().all(), "Há fluxo do dataset sem ficha no baseline_por_fluxo.csv"
for c in cols_ficha:
    assert np.allclose(cmp[c], cmp[c + "_baseline"], rtol=0, atol=1e-6), f"{c} difere do baseline"
print("Ficha das linhas == baseline_por_fluxo.csv: OK")

assert STATUS_ATUAL not in FEATURES and ALVO not in FEATURES
assert not set(FEATURES) & set(NAO_PODEM_ENTRAR), "Há coluna proibida entre as features"
print("Colunas da floresta:", ", ".join(FEATURES))
print("Alvo:", ALVO, "| status atual (só persistência):", STATUS_ATUAL)

## 2. Features de histórico e alvo futuro (iguais às da Tarefa 4)

- `z_robusto_t_menos_1` e `delta_z_robusto` usam só a medição anterior do mesmo fluxo (passado).
- `status_futuro` é a classe da 3ª medição seguinte. Só valem pares entre 11 e 13 minutos.
- O alvo é calculado **dentro de cada bloco**, então nenhum rótulo atravessa a fronteira entre treino, validação e teste.

O cálculo é o mesmo da Tarefa 4 de propósito (inclusive o `shift(1)` sem checar lacunas), para que o modelo medido seja o que foi validado.

In [ ]:
def preparar_historico(partes):
    """Junta os blocos disponíveis, ordena por fluxo e cria o lag causal."""
    quadros = []
    for nome, d in partes.items():
        d = d.copy().reset_index(drop=True)
        d["timestamp"] = pd.to_numeric(d["timestamp"], errors="raise")
        d["particao"] = nome
        quadros.append(d)
    h = pd.concat(quadros, ignore_index=True).sort_values(["fluxo_id", "timestamp"]).copy()
    g = h.groupby("fluxo_id", sort=False)
    h["z_robusto_t_menos_1"] = g["z_robusto"].shift(1)
    h["delta_z_robusto"] = h["z_robusto"] - h["z_robusto_t_menos_1"]
    return h


def separar(h, nome):
    return h[h["particao"] == nome].drop(columns="particao").copy()


def adicionar_alvo_futuro(df):
    df = df.sort_values(["fluxo_id", "timestamp"]).copy()
    g = df.groupby("fluxo_id", sort=False)
    df["status_futuro"] = g["classe"].shift(-PASSOS_FUTUROS)
    df["timestamp_futuro"] = g["timestamp"].shift(-PASSOS_FUTUROS)
    df["horizonte_observado_segundos"] = df["timestamp_futuro"] - df["timestamp"]
    dentro = df["horizonte_observado_segundos"].between(
        HORIZONTE_SEGUNDOS - TOLERANCIA_HORIZONTE_SEGUNDOS,
        HORIZONTE_SEGUNDOS + TOLERANCIA_HORIZONTE_SEGUNDOS)
    return df.loc[dentro & df["status_futuro"].notna()].copy()


def avaliar(y_real, y_prev, so_presentes=False):
    """Matriz 3x3 em contagem, precisão/recall/F1 por classe, F1 macro e acurácia balanceada."""
    y_real, y_prev = np.asarray(y_real), np.asarray(y_prev)
    rotulos = [c for c in CLASSES if (y_real == c).any()] if so_presentes else CLASSES
    mat = pd.DataFrame(confusion_matrix(y_real, y_prev, labels=CLASSES),
                       index=[f"real {c}" for c in CLASSES],
                       columns=[f"prevista {c}" for c in CLASSES])
    p, r, f, s = precision_recall_fscore_support(y_real, y_prev, labels=CLASSES, zero_division=0)
    por_classe = pd.DataFrame({"precisao": p, "recall": r, "F1": f, "medicoes": s}, index=CLASSES)
    return {"matriz": mat, "por_classe": por_classe,
            "f1_macro": float(f1_score(y_real, y_prev, labels=rotulos, average="macro", zero_division=0)),
            "acuracia_balanceada": float(balanced_accuracy_score(y_real, y_prev))}


print("Funções carregadas.")

## 3. Fluxos novos (fora do treino)

Os fluxos novos saem **inteiros** do treino e da validação. Eles têm ficha própria, calculada só no Período A (o mesmo piso de 1.500 medições com RTT da Tarefa 2), e o trecho avaliado nunca entra na mediana nem no MAD.

Regra automática (usa só treino e validação, **sem olhar o teste**):
- **longo estável:** mediana acima de `LIMIAR_LONGO_MS` com a maior proporção de OK;
- **curto degradado:** mediana abaixo de `LIMIAR_CURTO_MS`, com RISCO e FALHA presentes, e número de degradações mais próximo da mediana dos fluxos curtos (evita escolher o caso mais extremo).

Para escolher outros fluxos, preencha `FLUXOS_NOVOS_MANUAL` na célula 1. Se não houver fluxo que sirva, o enunciado manda escrever isso e **não forçar a conta**.

- [ ] O fluxo novo não aparece no treino
- [ ] A ficha vem só do Período A

In [ ]:
resumo = (pd.concat([tr, va], ignore_index=True).groupby("fluxo_id")
          .agg(rota=("rota", "first"), mediana=("mediana", "first"), linhas=("classe", "size"),
               n_ok=("classe", lambda s: int((s == "OK").sum())),
               n_risco=("classe", lambda s: int((s == "RISCO").sum())),
               n_falha=("classe", lambda s: int((s == "FALHA").sum())))
          .reset_index())
resumo["pct_ok"] = resumo["n_ok"] / resumo["linhas"]
resumo["n_degradacao"] = resumo["n_risco"] + resumo["n_falha"]


def ficha_valida(fluxo):
    linha = base[base["fluxo_id"] == fluxo]
    if linha.empty:
        return False
    if "status" in base.columns:
        return str(linha["status"].iloc[0]).strip().lower() == "ok"
    return True


resumo = resumo[resumo["fluxo_id"].map(ficha_valida)].copy()

if FLUXOS_NOVOS_MANUAL:
    novos = list(FLUXOS_NOVOS_MANUAL)
    faltam = [f for f in novos if f not in set(resumo["fluxo_id"])]
    assert not faltam, f"Fluxos sem ficha válida ou fora do dataset: {faltam}"
    criterio_escolha = "manual"
else:
    novos = []
    longos = resumo[resumo["mediana"] > LIMIAR_LONGO_MS].sort_values(["pct_ok", "linhas"], ascending=False)
    novos += longos["fluxo_id"].head(N_LONGO_ESTAVEL).tolist()
    curtos = resumo[(resumo["mediana"] < LIMIAR_CURTO_MS) & (resumo["n_risco"] > 0)
                    & (resumo["n_falha"] > 0) & ~resumo["fluxo_id"].isin(novos)]
    if len(curtos):
        alvo_degr = curtos["n_degradacao"].median()
        ordem = curtos.assign(dist=(curtos["n_degradacao"] - alvo_degr).abs()).sort_values("dist")
        novos += ordem["fluxo_id"].head(N_CURTO_DEGRADADO).tolist()
    criterio_escolha = "automático (longo estável + curto degradado, só com treino e validação)"

if novos:
    print("Fluxos novos (critério:", criterio_escolha + ")")
    print(resumo[resumo["fluxo_id"].isin(novos)][["fluxo_id", "rota", "mediana", "linhas", "n_ok",
                                                  "n_risco", "n_falha"]].round(2).to_string(index=False))
    print("\nObservação: outros fluxos com a mesma origem ou o mesmo destino continuam no treino "
          "(caminhos parcialmente compartilhados). Declare isso nas limitações.")
else:
    print("Nenhum fluxo novo disponível com ficha válida. Escreva isso no diário e não force a conta.")

## 4. Floresta final: treino e congelamento

A floresta é a da Tarefa 4 (200 árvores, `max_depth=10`, `min_samples_leaf=10`, Gini, sem `class_weight`, semente 42), treinada **só no treino** e **sem os fluxos novos**. Não há busca de hiperparâmetros aqui.

A validação aparece só como referência. No fim, o notebook grava os parâmetros e uma assinatura do modelo: se você rodar de novo e o modelo mudar, o notebook para.

In [ ]:
historico_pre = preparar_historico({"treino": tr, "validacao": va})
tr_lag, va_lag = separar(historico_pre, "treino"), separar(historico_pre, "validacao")

tr_pred = adicionar_alvo_futuro(tr_lag[~tr_lag["fluxo_id"].isin(novos)])
va_pred = adicionar_alvo_futuro(va_lag[~va_lag["fluxo_id"].isin(novos)])
assert len(tr_pred) and len(va_pred), "Sem pares suficientes para o alvo de 12 minutos"
assert not set(novos) & set(tr_pred["fluxo_id"]), "Fluxo novo apareceu no treino"

X_tr, y_tr = tr_pred[FEATURES], tr_pred[ALVO]
X_va, y_va = va_pred[FEATURES], va_pred[ALVO]
print("Linhas elegíveis - treino:", milhar(len(tr_pred)), "| validação:", milhar(len(va_pred)),
      "| fluxos no treino:", tr_pred["fluxo_id"].nunique())
print("Classes no treino:", y_tr.value_counts().reindex(CLASSES).to_dict())
print("Classes na validação:", y_va.value_counts().reindex(CLASSES).to_dict())

floresta = RandomForestClassifier(random_state=SEMENTE, n_jobs=-1, **HIPERPARAMETROS).fit(X_tr, y_tr)
profundidades = [a.get_depth() for a in floresta.estimators_]
folhas = [a.get_n_leaves() for a in floresta.estimators_]
print("\nCritério:", HIPERPARAMETROS["criterion"], "| árvores:", HIPERPARAMETROS["n_estimators"],
      "| max_depth:", HIPERPARAMETROS["max_depth"], "| min_samples_leaf:", HIPERPARAMETROS["min_samples_leaf"],
      "| semente:", SEMENTE)
print("Profundidade média:", round(float(np.mean(profundidades)), 2),
      "| folhas médias por árvore:", round(float(np.mean(folhas)), 2))

# Referência na validação (não decide nada): floresta x persistência
pred_va = floresta.predict(X_va)
ref_flor = avaliar(y_va, pred_va)
ref_pers = avaliar(y_va, va_pred[STATUS_ATUAL])
print("\nReferência na validação (fluxos conhecidos):")
print(f"  Floresta:    F1 macro = {br(ref_flor['f1_macro'])} | acurácia balanceada = {br(ref_flor['acuracia_balanceada'])}")
print(f"  Persistência: F1 macro = {br(ref_pers['f1_macro'])} | acurácia balanceada = {br(ref_pers['acuracia_balanceada'])}")

# Congelamento: assinatura calculada sobre as probabilidades na validação
assinatura = hashlib.sha256(np.round(floresta.predict_proba(X_va), 8).tobytes()).hexdigest()
if ARQ_TRAVA.exists() and REFAZER_CONGELAMENTO:
    raise RuntimeError("O teste já foi medido: não refaça o congelamento.")
if ARQ_PARAMS.exists() and not REFAZER_CONGELAMENTO:
    anterior = json.loads(ARQ_PARAMS.read_text(encoding="utf-8"))
    if anterior.get("assinatura_validacao") != assinatura:
        raise RuntimeError("O modelo de agora é diferente do que foi congelado antes. Se a mudança foi "
                           "intencional e o teste ainda não foi medido, use REFAZER_CONGELAMENTO = True.")
    print("\nModelo idêntico ao congelado anteriormente: OK")
else:
    ARQ_PARAMS.write_text(json.dumps({
        "modelo": "RandomForestClassifier", **HIPERPARAMETROS, "random_state": SEMENTE,
        "features": FEATURES, "alvo": ALVO, "horizonte_segundos": HORIZONTE_SEGUNDOS,
        "fluxos_novos": novos, "criterio_escolha_fluxos_novos": criterio_escolha,
        "criterio_ganho_persistencia": CRITERIO_GANHO,
        "profundidade_media": round(float(np.mean(profundidades)), 4),
        "folhas_medias": round(float(np.mean(folhas)), 4),
        "assinatura_validacao": assinatura, "scikit_learn": sklearn.__version__,
        "congelado_em_utc": datetime.now(timezone.utc).isoformat(),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\nParâmetros congelados em", ARQ_PARAMS.name)
joblib.dump(floresta, PASTA_MODELOS / "floresta_final.joblib")

## 5. Importância das features e regras em português

Uma floresta não tem um conjunto único de regras. Duas formas de ler regras, ambas **ilustrativas**:

- **A. Árvore componente 1:** uma das 200 árvores. Fica longa (profundidade até 10), então só mostramos a melhor folha de cada classe.
- **B. Árvore substituta rasa (profundidade 3):** uma árvore simples treinada para **imitar as previsões da floresta**. A fidelidade (% de previsões iguais às da floresta na validação) diz quanto ela representa.

Use o mesmo critério aceito na Tarefa 4 e declare no diário que as regras são ilustrativas. **As regras são gravadas antes de abrir o teste.**

In [ ]:
print("Importância das features na floresta:")
print(pd.Series(floresta.feature_importances_, index=FEATURES).sort_values(ascending=False).round(4).to_string())


def folhas_com_caminho(arvore, nomes):
    t = arvore.tree_
    saida = []

    def descer(no, condicoes):
        if t.children_left[no] == t.children_right[no]:
            cont = t.value[no][0]
            saida.append((condicoes, cont / cont.sum(), int(t.n_node_samples[no])))
            return
        nome, lim = nomes[t.feature[no]], float(t.threshold[no])
        descer(t.children_left[no], condicoes + [(nome, "<=", lim)])
        descer(t.children_right[no], condicoes + [(nome, ">", lim)])

    descer(0, [])
    return saida


def condicoes_em_texto(condicoes):
    """Junta cortes repetidos da mesma coluna (mantém o limite mais apertado)."""
    faixas = {}
    for nome, op, lim in condicoes:
        baixo, alto = faixas.get(nome, (None, None))
        if op == "<=":
            alto = lim if alto is None else min(alto, lim)
        else:
            baixo = lim if baixo is None else max(baixo, lim)
        faixas[nome] = (baixo, alto)
    partes = []
    for nome, (baixo, alto) in faixas.items():
        if baixo is not None and alto is not None:
            partes.append(f"{baixo:.2f} < {nome} <= {alto:.2f}")
        elif alto is not None:
            partes.append(f"{nome} <= {alto:.2f}")
        else:
            partes.append(f"{nome} > {baixo:.2f}")
    return " e ".join(partes)


def regras_por_classe(arvore, nomes, classes):
    """A folha de maior suporte para cada classe, no formato 'Se ... então ...'."""
    folhas_ = folhas_com_caminho(arvore, nomes)
    texto = []
    for i, classe in enumerate(classes):
        candidatas = [f for f in folhas_ if int(np.argmax(f[1])) == i]
        if not candidatas:
            texto.append(f"(nenhuma folha com classe {classe})")
            continue
        cond, prop, n = max(candidatas, key=lambda f: f[2])
        se = condicoes_em_texto(cond) if cond else "sempre"
        texto.append(f"Se {se}, então {classe}  (folha com {milhar(n)} amostras; pureza {br(prop[i] * 100, 1)}%)")
    return texto


classes_flor = list(floresta.classes_)
regras_a = regras_por_classe(floresta.estimators_[0], FEATURES, classes_flor)
print("\nA. Árvore componente 1 (ilustrativa; melhor folha de cada classe):")
for r in regras_a:
    print(" -", r)

X_tr_cheio = tr_pred[FEATURES]
substituta = DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=SEMENTE)
substituta.fit(X_tr_cheio, floresta.predict(X_tr_cheio))
fidelidade = float((substituta.predict(X_va) == pred_va).mean())
regras_b = regras_por_classe(substituta, FEATURES, list(substituta.classes_))
print(f"\nB. Árvore substituta rasa (fidelidade à floresta na validação: {br(fidelidade * 100, 2)}%):")
for r in regras_b:
    print(" -", r)
print("\n", export_text(substituta, feature_names=FEATURES, class_names=list(substituta.classes_), decimals=2))

(PASTA_MODELOS / "floresta_final_regras.txt").write_text(
    "A. Árvore componente 1 (ilustrativa)\n" + "\n".join(regras_a) +
    f"\n\nB. Árvore substituta rasa (fidelidade {fidelidade:.4f})\n" + "\n".join(regras_b) + "\n",
    encoding="utf-8")

### Pare aqui antes de abrir o teste

Preencha o diário (seção 1 da Tarefa 5) com:

- parâmetros, colunas e semente (célula 4);
- as três regras em português e o aviso de que são ilustrativas (célula 5);
- os fluxos novos escolhidos e o critério (célula 3);
- o critério de ganho sobre a persistência (`CRITERIO_GANHO`).

Só depois mude `ABRIR_TESTE = True` na célula 1 e rode o notebook **inteiro uma vez**.

## 6. Teste temporal (uma vez)

Fluxos conhecidos, trecho mais recente do Período B. A floresta já está congelada. Esta célula lê o arquivo de teste, mede, grava os resultados e cria a trava.

- [ ] Matriz 3x3 com contagem
- [ ] Precisão, recall e F1 de OK, RISCO e FALHA, e F1 macro
- [ ] Recall de FALHA e a troca RISCO <-> FALHA
- [ ] Persistência na mesma passada
- [ ] Dois casos: um acerto em caminho longo estável (OK) e um erro relevante, com fluxo e timestamp

In [ ]:
res = None
if not ABRIR_TESTE:
    print("Teste FECHADO (ABRIR_TESTE = False). Nada foi lido do arquivo de teste.")
else:
    if ARQ_TRAVA.exists() and not FORCAR_REMEDIR:
        raise RuntimeError("O teste já foi medido (teste_medido.json existe). Ele só vale uma vez.")
    te = pd.read_csv(CAMINHO_TESTE)
    te["timestamp"] = pd.to_numeric(te["timestamp"], errors="raise")
    assert set(te["conjunto"]) == {"teste"} and set(te["periodo"]) == {"B"}
    lim_va_max = va.groupby("fluxo_id")["timestamp"].max()
    lim_te_min = te.groupby("fluxo_id")["timestamp"].min()
    comuns = lim_va_max.index.intersection(lim_te_min.index)
    assert (lim_va_max[comuns] < lim_te_min[comuns]).all(), "Validação e teste se misturam em algum fluxo"

    historico_final = preparar_historico({"treino": tr, "validacao": va, "teste": te})
    te_lag = separar(historico_final, "teste")
    te_pred = adicionar_alvo_futuro(te_lag[~te_lag["fluxo_id"].isin(novos)])
    assert len(te_pred), "Sem pares suficientes no teste"
    assert not set(te_pred["fluxo_id"]) & set(novos)

    X_te, y_te = te_pred[FEATURES], te_pred[ALVO]
    pred_te = floresta.predict(X_te)
    pers_te = te_pred[STATUS_ATUAL].to_numpy()
    te_pred = te_pred.assign(previsao=pred_te, persistencia=pers_te)

    r_flor, r_pers = avaliar(y_te, pred_te), avaliar(y_te, pers_te)
    print("--- TESTE ÚNICO: CLASSE 12 MINUTOS À FRENTE (fluxos conhecidos) ---")
    print("Linhas do teste:", milhar(len(te_pred)), "| fluxos:", te_pred["fluxo_id"].nunique())
    print("Classes reais:", y_te.value_counts().reindex(CLASSES).to_dict())
    print("\nMatriz de confusão - floresta (contagem):")
    print(r_flor["matriz"].to_string())
    print("\nMétricas por classe - floresta:")
    print(r_flor["por_classe"].round(4).to_string())
    print("\nMatriz de confusão - persistência (contagem):")
    print(r_pers["matriz"].to_string())
    print("\nMétricas por classe - persistência:")
    print(r_pers["por_classe"].round(4).to_string())

    rec_falha = float(r_flor["por_classe"].loc["FALHA", "recall"])
    rec_falha_p = float(r_pers["por_classe"].loc["FALHA", "recall"])
    rec_risco = float(r_flor["por_classe"].loc["RISCO", "recall"])
    rec_risco_p = float(r_pers["por_classe"].loc["RISCO", "recall"])
    m = r_flor["matriz"]
    troca_rf = int(m.loc["real RISCO", "prevista FALHA"])
    troca_fr = int(m.loc["real FALHA", "prevista RISCO"])
    falha_ok = int(m.loc["real FALHA", "prevista OK"])
    print("\n--- COMPARAÇÃO NO MESMO CONJUNTO ---")
    print(f"Floresta:     F1 macro = {br(r_flor['f1_macro'])} | acurácia balanceada = {br(r_flor['acuracia_balanceada'])}"
          f" | recall FALHA = {br(rec_falha)} | recall RISCO = {br(rec_risco)}")
    print(f"Persistência: F1 macro = {br(r_pers['f1_macro'])} | acurácia balanceada = {br(r_pers['acuracia_balanceada'])}"
          f" | recall FALHA = {br(rec_falha_p)} | recall RISCO = {br(rec_risco_p)}")
    ganho = r_flor[CRITERIO_GANHO] - r_pers[CRITERIO_GANHO]
    print(f"Diferença em {CRITERIO_GANHO} (floresta - persistência): {ganho:+.4f}")
    print(f"\nTroca RISCO <-> FALHA: real RISCO previsto FALHA = {troca_rf} | real FALHA previsto RISCO = {troca_fr}")
    print(f"FALHAs reais previstas como OK: {falha_ok} de {int(m.loc['real FALHA'].sum())}")

    erros = te_pred[te_pred[ALVO] != te_pred["previsao"]]
    print("\nErros por par (real -> previsto):")
    print(erros.groupby([ALVO, "previsao"]).size().sort_values(ascending=False).to_string())
    print("Erros em caminho curto (mediana < 5 ms):", int((erros["mediana"] < LIMIAR_CURTO_MS).sum()),
          "| em caminho longo (mediana > 150 ms):", int((erros["mediana"] > LIMIAR_LONGO_MS).sum()))

    # ---- Dois casos concretos ----
    if "datetime_utc" not in te_pred.columns:
        te_pred["datetime_utc"] = pd.to_datetime(te_pred["timestamp"], unit="s", utc=True)
    te_pred["instante_alvo_utc"] = pd.to_datetime(te_pred["timestamp_futuro"], unit="s", utc=True)
    cols_caso = [c for c in ["fluxo_id", "datetime_utc", "instante_alvo_utc", "mediana", "avg", "z_robusto",
                             "perda_pct", "n5_risco", STATUS_ATUAL, ALVO, "previsao", "persistencia"]
                 if c in te_pred.columns]

    acerto = te_pred[(te_pred[ALVO] == "OK") & (te_pred["previsao"] == "OK")
                     & (te_pred["mediana"] > LIMIAR_LONGO_MS) & (te_pred["z_robusto"].abs() < 2)
                     & (te_pred["perda_pct"] == 0)].sort_values(["mediana", "fluxo_id", "timestamp"],
                                                               ascending=[False, True, True])
    print("\nCaso 1 - acerto em caminho longo estável (OK):")
    if len(acerto):
        print(acerto.head(1)[cols_caso].to_string(index=False))
    else:
        print("Não apareceu no teste.")

    erro = None
    for real, prev in [("FALHA", "OK"), ("FALHA", "RISCO"), ("RISCO", "OK"), ("RISCO", "FALHA"),
                       ("OK", "FALHA"), ("OK", "RISCO")]:
        cand = te_pred[(te_pred[ALVO] == real) & (te_pred["previsao"] == prev)]
        if len(cand):
            erro = cand.sort_values(["fluxo_id", "timestamp"]).head(1)
            print(f"\nCaso 2 - erro relevante (real {real} -> previsto {prev}; {len(cand)} ocorrências desse par):")
            print(erro[cols_caso].to_string(index=False))
            break
    if erro is None:
        print("\nCaso 2: não houve erros no teste.")

    res = {
        "linhas_teste": int(len(te_pred)),
        "matriz_floresta": r_flor["matriz"].astype(int).to_dict(),
        "por_classe_floresta": r_flor["por_classe"].round(6).to_dict(),
        "f1_macro_floresta": r_flor["f1_macro"], "acuracia_balanceada_floresta": r_flor["acuracia_balanceada"],
        "recall_falha_floresta": rec_falha, "recall_risco_floresta": rec_risco,
        "f1_macro_persistencia": r_pers["f1_macro"], "acuracia_balanceada_persistencia": r_pers["acuracia_balanceada"],
        "recall_falha_persistencia": rec_falha_p, "recall_risco_persistencia": rec_risco_p,
        "criterio_ganho": CRITERIO_GANHO, "ganho": float(ganho),
        "troca_risco_para_falha": troca_rf, "troca_falha_para_risco": troca_fr,
        "falha_prevista_ok": falha_ok,
        "medido_em_utc": datetime.now(timezone.utc).isoformat(),
    }
    (PASTA_MODELOS / "resultado_teste.json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
    ARQ_TRAVA.write_text(json.dumps({"medido_em_utc": res["medido_em_utc"],
                                     "aviso": "Teste medido. Não meça de novo."}, ensure_ascii=False), encoding="utf-8")
    print("\nResultados salvos e trava criada (teste_medido.json).")

## 7. Fluxos que a floresta não viu

Os fluxos novos são avaliados no **Período B inteiro** (treino + validação + teste, juntos e em ordem), porque a floresta nunca viu nenhuma linha deles. A ficha de cada um é a do Período A, calculada antes do trecho avaliado.

- [ ] O fluxo novo não aparece no treino
- [ ] Mediana e MAD não usam o trecho avaliado
- [ ] Um caminho longo estável permaneceu OK? Uma degradação foi marcada RISCO ou FALHA?

In [ ]:
resultado_novos = None
if not ABRIR_TESTE or res is None:
    print("Aguardando o teste (ABRIR_TESTE = True).")
elif not novos:
    print("Não havia fluxo sobrando com ficha válida. Escreva isso no diário; não force a conta.")
else:
    assert not set(novos) & set(tr_pred["fluxo_id"]), "Fluxo novo aparece no treino"
    for fl in novos:
        assert ficha_valida(fl), f"{fl} sem ficha válida no baseline_por_fluxo.csv"
    hist_novos = historico_final[historico_final["fluxo_id"].isin(novos)].drop(columns="particao")
    nov_pred = adicionar_alvo_futuro(hist_novos)
    assert len(nov_pred), "Sem pares suficientes nos fluxos novos"
    prev_n = floresta.predict(nov_pred[FEATURES])
    nov_pred = nov_pred.assign(previsao=prev_n)

    linhas = []
    for fl, g in nov_pred.groupby("fluxo_id"):
        real, prev = g[ALVO], g["previsao"]
        degr = real.isin(["RISCO", "FALHA"])
        detect = degr & prev.isin(["RISCO", "FALHA"])
        linhas.append({"fluxo_id": fl, "mediana_ms": round(float(g["mediana"].iloc[0]), 2),
                       "linhas": len(g), "real_OK_%": round(float((real == "OK").mean() * 100), 1),
                       "prevista_OK_%": round(float((prev == "OK").mean() * 100), 1),
                       "degradações_reais": int(degr.sum()), "degradações_detectadas": int(detect.sum()),
                       "erros": int((real != prev).sum())})
    tabela_novos = pd.DataFrame(linhas)
    print("Resumo por fluxo novo (alvo = classe 12 min à frente):")
    print(tabela_novos.to_string(index=False))

    r_nov = avaliar(nov_pred[ALVO], prev_n, so_presentes=True)
    r_nov_p = avaliar(nov_pred[ALVO], nov_pred[STATUS_ATUAL], so_presentes=True)
    print("\nMatriz de confusão nos fluxos novos (contagem):")
    print(r_nov["matriz"].to_string())
    print(f"F1 macro (só classes presentes): floresta = {br(r_nov['f1_macro'])} | persistência = {br(r_nov_p['f1_macro'])}")

    print("\nLeitura:")
    for _, linha in tabela_novos.iterrows():
        if linha["mediana_ms"] > LIMIAR_LONGO_MS:
            print(f" - {linha['fluxo_id']} (caminho longo): a floresta previu OK em {linha['prevista_OK_%']}% das linhas "
                  f"(real OK: {linha['real_OK_%']}%).")
        if linha["degradações_reais"] > 0:
            print(f" - {linha['fluxo_id']}: {linha['degradações_detectadas']} de {linha['degradações_reais']} degradações reais "
                  f"foram marcadas RISCO ou FALHA.")
    resultado_novos = {"fluxos": novos, "f1_macro": r_nov["f1_macro"], "f1_macro_persistencia": r_nov_p["f1_macro"],
                       "resumo": tabela_novos.to_dict(orient="records")}
    (PASTA_MODELOS / "resultado_fluxos_novos.json").write_text(
        json.dumps(resultado_novos, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

## 8. O que declarar e ficha da árvore

A célula abaixo monta o veredito pelo critério declarado antes do teste e imprime a ficha para colar no diário. Os textos de limitações vêm logo depois.

In [ ]:
if res is None:
    print("Aguardando o teste (ABRIR_TESTE = True).")
else:
    ganho = res["ganho"]
    print("--- VEREDITO (critério declarado antes do teste:", res["criterio_ganho"] + ") ---")
    if ganho > 0:
        print(f"A floresta supera a persistência em {res['criterio_ganho']} ({ganho:+.4f}). Pode ser chamada de preditor, com as ressalvas abaixo.")
    else:
        print(f"Sem ganho sobre a persistência em {res['criterio_ganho']} ({ganho:+.4f}). O entregável é um DETECTOR do estado atual, não um preditor.")
    if res["recall_falha_floresta"] < res["recall_falha_persistencia"]:
        print("Ressalva: o recall de FALHA da floresta é MENOR que o da persistência "
              f"({br(res['recall_falha_floresta'])} contra {br(res['recall_falha_persistencia'])}): ela deixa passar mais FALHAs.")
    if res["acuracia_balanceada_floresta"] < res["acuracia_balanceada_persistencia"]:
        print("Ressalva: a acurácia balanceada da floresta é MENOR que a da persistência.")

    print("\n--- FICHA DA ÁRVORE (para colar no diário) ---")
    ficha = {
        "Problema": "Classe OK, RISCO ou FALHA pelo desvio ao baseline do fluxo (alvo: 12 minutos à frente)",
        "Unidade": "Uma medição de um fluxo_id no Período B",
        "Baseline": "Período A, mediana e MAD, piso de 1.500 RTT válidos",
        "Colunas": ", ".join(FEATURES),
        "Modelo": (f"Floresta aleatória: {HIPERPARAMETROS['n_estimators']} árvores, critério {HIPERPARAMETROS['criterion']}, "
                   f"max_depth {HIPERPARAMETROS['max_depth']}, min_samples_leaf {HIPERPARAMETROS['min_samples_leaf']}, "
                   f"profundidade média {round(float(np.mean(profundidades)), 2)}, folhas médias {round(float(np.mean(folhas)), 1)}, "
                   f"semente {SEMENTE}"),
        "Teste (uma vez)": f"F1 macro {br(res['f1_macro_floresta'])}; recall de FALHA {br(res['recall_falha_floresta'])}",
        "Persistência (mesmo teste)": f"F1 macro {br(res['f1_macro_persistencia'])}; recall de FALHA {br(res['recall_falha_persistencia'])}",
        "Fluxos novos": ", ".join(f.replace("|", "\\|") for f in novos) if novos else "nenhum com baseline suficiente",
        "O que ela não faz": "Não classifica distância. Não opera em fluxo sem ficha.",
        "Como reproduzir": "requirements.txt, este notebook, config/ (semente 42)",
    }
    for k, v in ficha.items():
        print(f"| {k} | {v} |")

print("\n--- COLUNAS PARA O DICIONÁRIO v0.4 (iguais às usadas) ---")
origem = {"z_robusto_t_menos_1": "derivada no notebook: z_robusto da medição anterior do fluxo",
          "delta_z_robusto": "derivada no notebook: z_robusto atual menos o da medição anterior",
          "z_robusto": "Tarefa 2: (RTT atual - mediana) / (1,4826 x MAD)",
          "perda_pct": "Tarefa 2: (enviados - recebidos) / enviados x 100",
          "jitter_relativo": "Tarefa 2: jitter atual / jitter_tipico",
          "n5_risco": "Tarefa 2: nº das últimas 5 medições que cumprem o critério de RISCO (linha 5)",
          "n5_timeout": "Tarefa 2: timeouts nas últimas 5 medições, incluindo a atual",
          "n5_aumento80": "Tarefa 2: nº das últimas 5 com aumento_pct > 80",
          "status_futuro": "ALVO: classe da 3ª medição seguinte (~12 min) do mesmo fluxo"}
for c in FEATURES + [ALVO]:
    print(f"| {c} | {origem[c]} |")

### Limitações (para a seção 4 do diário)

- **A floresta aprende a política de rotulagem.** O rótulo foi feito com as mesmas métricas que ela recebe. Ela não aprende um ticket de roteador nem um evento real de rede.
- **Detector ou preditor:** vale o veredito da célula 8, calculado com o critério declarado antes do teste, junto com as ressalvas de recall de FALHA e acurácia balanceada.
- **Baseline fixo não acompanha troca de rota.** Se o caminho muda depois do Período A, o desvio vira "degradação".
- **Rajada curta estima mal o jitter.** Cada medição tem poucos pacotes (em 3 pacotes, 1 perda já é 33%), então o jitter e a perda são grosseiros.
- **Perda de um único pacote já é FALHA** pela tabela de rótulo (perda >= 10%), o que pesa muito em rotas estáveis.
- **Rotas curtas têm MAD minúsculo.** Desvios de décimos de milissegundo geram `z_robusto` alto.
- **O teste cobre o fim do período (inclui fim de semana)**, enquanto o treino é majoritariamente dia útil.
- **Fluxos novos compartilham origem ou destino com fluxos do treino**, então a independência é parcial.
- **Anchors do RIPE Atlas não são a rede do campus.**
- **As regras em português são ilustrativas:** a previsão oficial é a votação das 200 árvores.

### Checklist final

- [ ] A profundidade não foi escolhida de novo olhando o teste
- [ ] O baseline do teste não usou medição do próprio teste
- [ ] País, IP, rota e RTT absoluto não estão na floresta
- [ ] Matriz 3x3 em contagem, F1 por classe e F1 macro no diário
- [ ] Dois casos (acerto longo estável e erro relevante) com fluxo e timestamp
- [ ] Fluxos novos e resultado no diário
- [ ] Ficha da árvore e dicionário v0.4 preenchidos
- [ ] Board atualizado e diário de todos